In [2]:
pip install nvidia-cublas-cu12 nvidia-cudnn-cu12

Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install faster-whisper soundfile nvidia-cublas-cu12 nvidia-cudnn-cu12

Note: you may need to restart the kernel to use updated packages.


In [5]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
from faster_whisper import WhisperModel

model = model = WhisperModel("large-v3", device="cuda", compute_type="float16")
print("모델 로드 완료")

모델 로드 완료


In [20]:
INPUT = r"D:\ai\refs\ko_main -1.wav"

segments, info = model.transcribe(INPUT, language="ko")
print(f"총 길이: {info.duration:.0f}초, 전사 시작...\n")

all_segs = []
for seg in segments:  # 제너레이터라 여기서 실제 전사가 돌면서 한 줄씩 찍힘
    all_segs.append(seg)
    print(f"{seg.start:6.1f}s ~ {seg.end:6.1f}s | {seg.text.strip()}")

print(f"\n전사 완료: {len(all_segs)}개 구간")

총 길이: 511초, 전사 시작...

   0.0s ~    6.7s | 지휘관 씨, 왔어요?
   7.8s ~    9.4s | 히힛, 기다렸어요
   9.4s ~   16.0s | 오늘 일은 조금만 더 하면 끝나니까
  16.0s ~   18.7s | 여기 앉아서 먼저 하고 있어요
  18.7s ~   29.3s | 뭐니뭐니 해도 오늘은 지휘관 씨와 붐을 하는 날이니까요
  29.3s ~   32.4s | 얼른 일을 끝내고 싶었는데
  32.4s ~   35.3s | 급한 의뢰가 들어왔지 뭐예요
  35.3s ~   38.3s | 오, 켰어요?
  39.9s ~   44.7s | 그러고 보니 케이블이 낡아서 접촉 분량이 있었던 것 같네요
  44.7s ~   48.3s | 잠깐 실례할게요
  48.3s ~   54.3s | 이 모니터는 거의 안 썼으니까요
  55.8s ~   57.6s | 음...
  57.6s ~   59.3s | 으... 아...
  59.3s ~   63.7s | 혹시 케이블이 아니라 포트 문제인 걸까요?
  65.1s ~   68.4s | 지휘관 씨, 이걸 그쪽 포트에 꽂아볼래요?
  69.4s ~   70.9s | 이러면 불이 들어올 거예요
  70.9s ~   78.6s | 히히, 지휘관 씨가 붐 가위를 조작하는 모습을
  78.6s ~   80.7s | 이렇게 가까이서 볼 수 있다니
  80.7s ~   84.0s | 평소에 착하게 산 보람이 있네요
  84.0s ~   86.4s | 최고로 행복해요
  89.3s ~   97.3s | director志ный님 Tsuzumi
  97.3s ~  101.8s | 음. 일을 하러 가야 하는데...
 101.8s ~  104.8s | 저도 모르게 빠져드네요
 104.8s ~  107.3s | 오... 오...
 107.3s ~  109.5s | 와하!
 109.5s ~  110.6s | 흠흠...
 110.6s ~  112.3s | 으르르 훌륭해요
 112.3s ~  116.1s 

In [21]:
candidates = []
print("=== 참조 후보 (3~10초) ===")
for seg in all_segs:
    dur = seg.end - seg.start
    text = seg.text.strip()
    if 3.0 <= dur <= 10.0 and text:
        candidates.append((seg.start, seg.end, text))
        print(f"[{len(candidates)-1}] {seg.start:6.1f}s ~ {seg.end:6.1f}s ({dur:.1f}s) | {text}")

=== 참조 후보 (3~10초) ===
[0]    0.0s ~    6.7s (6.7s) | 지휘관 씨, 왔어요?
[1]    9.4s ~   16.0s (6.6s) | 오늘 일은 조금만 더 하면 끝나니까
[2]   29.3s ~   32.4s (3.1s) | 얼른 일을 끝내고 싶었는데
[3]   35.3s ~   38.3s (3.0s) | 오, 켰어요?
[4]   39.9s ~   44.7s (4.8s) | 그러고 보니 케이블이 낡아서 접촉 분량이 있었던 것 같네요
[5]   44.7s ~   48.3s (3.6s) | 잠깐 실례할게요
[6]   48.3s ~   54.3s (6.0s) | 이 모니터는 거의 안 썼으니까요
[7]   59.3s ~   63.7s (4.3s) | 혹시 케이블이 아니라 포트 문제인 걸까요?
[8]   65.1s ~   68.4s (3.3s) | 지휘관 씨, 이걸 그쪽 포트에 꽂아볼래요?
[9]   70.9s ~   78.6s (7.7s) | 히히, 지휘관 씨가 붐 가위를 조작하는 모습을
[10]   80.7s ~   84.0s (3.3s) | 평소에 착하게 산 보람이 있네요
[11]   89.3s ~   97.3s (8.0s) | director志ный님 Tsuzumi
[12]   97.3s ~  101.8s (4.5s) | 음. 일을 하러 가야 하는데...
[13]  112.3s ~  116.1s (3.8s) | 으, 으앙우우으😄
[14]  119.2s ~  123.2s (4.0s) | 그럼 저는 일하러 돌아가 볼게요
[15]  125.6s ~  129.5s (3.9s) | 아니, 별건 아니에요
[16]  129.5s ~  134.5s (5.0s) | 빨리 대처하지 않으면 방주 전체 인프라에 문제가 생기거든요
[17]  134.5s ~  139.3s (4.8s) | 슈엔 씨가 부탁한 거라면 파업해버리겠지만
[18]  139.3s ~  145.5s (6.2s) | 시민들의 삶에 관련된 일이면 그럴 수 없으니까요
[19]  145

In [9]:
import soundfile as sf
from IPython.display import Audio

IDX = 0  # <- 원하는 후보 번호로 바꾸고 실행

start, end, text = candidates[IDX]
data, sr = sf.read(INPUT)
clip = data[int(start * sr):int(end * sr)]
if clip.ndim > 1:
    clip = clip.mean(axis=1)

out_path = INPUT.rsplit(".", 2)[0] + "_ref.wav"
sf.write(out_path, clip, sr)
print(f"저장: {out_path}")
print(f"prompt_text: {text}")

Audio(clip, rate=sr)  # 노트북에서 바로 재생해서 확인

저장: D:\ai\refs\ko_main -1_ref_ref.wav
prompt_text: 오늘 일은 조금만 더 하면 끝나니까


In [31]:
# TTS 테스트 셀
import requests, io
import soundfile as sf
from IPython.display import Audio

KO_MAIN = r"D:\ai\refs\ko_main_ref32.wav"
KO_MAIN_TEXT = "지휘관 씨, 왔어요?"   # 전사에서 나온 그 문장
KO_SUB = r"D:\ai\refs\ko_sub_ref32.wav"

def tts_test(text, use_aux=False):
    params = {
        "text": text,
        "text_lang": "ko",
        "ref_audio_path": KO_MAIN,
        "prompt_text": KO_MAIN_TEXT,
        "prompt_lang": "ko",
    }
    if use_aux:
        params["aux_ref_audio_paths"] = [KO_SUB]
    r = requests.get("http://127.0.0.1:9880/tts", params=params, timeout=120)
    r.raise_for_status()
    data, sr = sf.read(io.BytesIO(r.content))
    return Audio(data, rate=sr)

# A: 메인 단독
tts_test("안녕하세요, 오늘 방송 시작해볼게요. 다들 저녁은 드셨나요?")

In [14]:
import soundfile as sf

for f in [r"D:\ai\refs\ko_main -1_ref.wav", r"D:\ai\refs\ko_sub 1_ref.wav"]:
    info = sf.info(f)
    print(f, "|", info.samplerate, "Hz |", round(info.duration, 2), "초 |", info.channels, "ch")

D:\ai\refs\ko_main -1_ref.wav | 96000 Hz | 6.6 초 | 1 ch
D:\ai\refs\ko_sub 1_ref.wav | 48000 Hz | 7.7 초 | 1 ch


In [15]:
# 참조 음성 정리: 32kHz 리샘플 + 공백 없는 파일명으로 재저장
import soundfile as sf
import numpy as np
from scipy.signal import resample_poly

TARGET_SR = 32000

jobs = [
    (r"D:\ai\refs\ko_main -1_ref.wav", r"D:\ai\refs\ko_main_ref32.wav"),
    (r"D:\ai\refs\ko_sub 1_ref.wav",  r"D:\ai\refs\ko_sub_ref32.wav"),
]

for src, dst in jobs:
    data, sr = sf.read(src)
    if data.ndim > 1:
        data = data.mean(axis=1)
    g = np.gcd(sr, TARGET_SR)
    data = resample_poly(data, TARGET_SR // g, sr // g)
    sf.write(dst, data.astype(np.float32), TARGET_SR, subtype="PCM_16")
    info = sf.info(dst)
    print(dst, "|", info.samplerate, "Hz |", round(info.duration, 2), "초")

D:\ai\refs\ko_main_ref32.wav | 32000 Hz | 6.6 초
D:\ai\refs\ko_sub_ref32.wav | 32000 Hz | 7.7 초


In [30]:
# 개선된 자르기 셀: 앞뒤 무음 제거 + 실발화 길이 검증
import numpy as np
import soundfile as sf
from IPython.display import Audio

INPUT = r"D:\ai\refs\ko_main -1.wav"   # 원본 (긴 파일)
IDX = 0                                # 후보 번호
PAD = 1.0                               # 구간 앞뒤 여유 (타임스탬프 오차 대비)

start, end, text = candidates[IDX]
data, sr = sf.read(INPUT)
if data.ndim > 1:
    data = data.mean(axis=1)

# 여유 포함해서 자르기
s = max(0, int((start - PAD) * sr))
e = min(len(data), int((end + PAD) * sr))
clip = data[s:e]

# 에너지 기반 무음 트리밍
win = int(sr * 0.05)
rms = np.array([np.sqrt(np.mean(clip[i:i+win]**2)) for i in range(0, len(clip)-win, win)])
th = rms.max() * 0.05  # 최대 에너지의 5% 미만 = 무음
voiced = np.where(rms > th)[0]
clip = clip[voiced[0]*win : (voiced[-1]+1)*win]

dur = len(clip) / sr
print(f"실발화 길이: {dur:.2f}초", "✅ OK" if 3.0 <= dur <= 10.0 else "❌ 3~10초 벗어남 - 다른 구간 선택")
print(f"대사: {text}")

if 3.0 <= dur <= 10.0:
    # 32kHz 리샘플 + 저장까지 한 번에
    from scipy.signal import resample_poly
    g = np.gcd(sr, 32000)
    clip32 = resample_poly(clip, 32000 // g, sr // g)
    out = r"D:\ai\refs\ko_main_ref32.wav"
    sf.write(out, clip32.astype(np.float32), 32000, subtype="PCM_16")
    print(f"저장: {out}")

Audio(clip, rate=sr)  # 잘린 결과 직접 확인

실발화 길이: 1.55초 ❌ 3~10초 벗어남 - 다른 구간 선택
대사: 지휘관 씨, 왔어요?


In [28]:
# 최종 참조 파일 검증: 실제 내용 재전사
REF_FILE = r"D:\ai\refs\ko_main_ref32.wav"

segs, _ = model.transcribe(REF_FILE, language="ko")
actual_text = " ".join(s.text.strip() for s in segs)
print(f"실제 담긴 내용: {actual_text}")
# -> 이 출력을 그대로 KO_MAIN_TEXT로 사용

실제 담긴 내용: 음, 그러고 보니 케이블이 낡아서 접촉 분량이 있었던 것 같네요.
